In [1]:
import os, sys, time
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
sys.path.insert(0, '..')

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName('week2-optimization')
    .master('local[*]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.session.timeZone', 'America/New_York')
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '4g')
    .config('spark.sql.autoBroadcastJoinThreshold', '104857600')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print('Spark ready:', spark.version)

Spark ready: 3.5.9


In [2]:
# -- change this to your absolute path --
BASE = r'f:\Sem 3A\Data-intensive Computing\urban-data-platform\data'

trips = spark.read.format('delta').load(f'{BASE}/gold/integrated_taxi_trips')
trips.createOrReplaceTempView('trips')

print('Views ready. Trip count:', trips.count())

Views ready. Trip count: 8480836


In [4]:
trips.printSchema()

root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- pickup_hour: timestamp (nullable = true

In [ ]:
#q1
spark.sql("""
    SELECT month, pickup_location_id, pickup_zone, COUNT(*) AS taxi_demand
    FROM trips
    WHERE year > 2023
    GROUP BY month, pickup_location_id, pickup_zone
    ORDER BY taxi_demand DESC
""").show(20)


+-----+------------------+--------------------+-----------+
|month|pickup_location_id|         pickup_zone|taxi_demand|
+-----+------------------+--------------------+-----------+
|    3|               132|         JFK Airport|     148208|
|    3|               161|      Midtown Center|     146328|
|    3|               237|Upper East Side S...|     142506|
|    2|               161|      Midtown Center|     137095|
|    1|               132|         JFK Airport|     137002|
|    1|               237|Upper East Side S...|     135472|
|    1|               161|      Midtown Center|     134981|
|    2|               237|Upper East Side S...|     132635|
|    3|               236|Upper East Side N...|     130177|
|    1|               236|Upper East Side N...|     128108|
|    2|               236|Upper East Side N...|     124337|
|    2|               132|         JFK Airport|     119522|
|    3|               162|        Midtown East|     112693|
|    3|               230|Times Sq/Theat

In [20]:
trips.select("condition_code").distinct().orderBy("condition_code").show()


+--------------+
|condition_code|
+--------------+
|          NULL|
|             1|
|             2|
|             3|
|             4|
|             5|
|             7|
|             8|
|             9|
|            10|
|            12|
|            13|
|            14|
|            15|
|            16|
+--------------+



Likely "Meteostat" code

In [ ]:
#q2
spark.sql("""
SELECT
    CASE condition_code
        WHEN 1 THEN 'Clear'
        WHEN 2 THEN 'Fair'
        WHEN 3 THEN 'Cloudy'
        WHEN 4 THEN 'Overcast'
        WHEN 5 THEN 'Fog'
        WHEN 6 THEN 'Freezing Fog'
        WHEN 7 THEN 'Light Rain'
        WHEN 8 THEN 'Rain'
        WHEN 9 THEN 'Heavy Rain'
        WHEN 10 THEN 'Freezing Rain'
        WHEN 11 THEN 'Heavy Freezing Rain'
        WHEN 12 THEN 'Sleet'
        WHEN 13 THEN 'Heavy Sleet'
        WHEN 14 THEN 'Light Snowfall'
        WHEN 15 THEN 'Snowfall'
        WHEN 16 THEN 'Heavy Snowfall'
        ELSE 'Unknown'
    END AS weather_condition,
    COUNT(*) AS trip_count,
    ROUND(AVG(trip_distance), 3) AS avg_trip_distance
FROM trips
WHERE trip_distance > 0
GROUP BY condition_code
ORDER BY avg_trip_distance DESC
""").show()


+-----------------+----------+-----------------+
|weather_condition|trip_count|avg_trip_distance|
+-----------------+----------+-----------------+
|            Clear|    207482|            3.637|
|             Fair|   2742193|            3.536|
|         Overcast|   1922398|            3.511|
|           Cloudy|   2002764|            3.385|
|              Fog|    244105|            3.359|
|       Heavy Rain|    260892|             3.33|
|   Heavy Snowfall|     20079|             3.32|
|       Light Rain|    628442|            3.255|
|          Unknown|        18|            3.217|
|   Light Snowfall|    167467|            3.151|
|         Snowfall|     29851|             3.12|
|             Rain|    208650|            3.104|
|            Sleet|     18343|            3.026|
|      Heavy Sleet|     15881|            2.976|
|    Freezing Rain|     12271|             2.94|
+-----------------+----------+-----------------+



In [21]:
#q3
spark.sql("""
SELECT
    CASE
        WHEN pm25_hourly_avg < 10 THEN '0-10'
        WHEN pm25_hourly_avg < 20 THEN '10-20'
        WHEN pm25_hourly_avg < 30 THEN '20-30'
        WHEN pm25_hourly_avg < 50 THEN '30-50'
        ELSE '50+'
    END AS pm25_range,
    COUNT(*) AS taxi_demand,
    ROUND(AVG(trip_distance), 2) AS avg_trip_distance
FROM trips
WHERE pm25_hourly_avg IS NOT NULL
GROUP BY
    CASE
        WHEN pm25_hourly_avg < 10 THEN '0-10'
        WHEN pm25_hourly_avg < 20 THEN '10-20'
        WHEN pm25_hourly_avg < 30 THEN '20-30'
        WHEN pm25_hourly_avg < 50 THEN '30-50'
        ELSE '50+'
    END
ORDER BY pm25_range
""").show()


+----------+-----------+-----------------+
|pm25_range|taxi_demand|avg_trip_distance|
+----------+-----------+-----------------+
|      0-10|    6247615|             3.49|
|     10-20|    1478303|             3.32|
|     20-30|     515687|             3.18|
|     30-50|     199203|             3.41|
+----------+-----------+-----------------+



In [24]:
#q4
spark.sql("""
WITH weather_demand AS (
    SELECT
        pickup_zone,
        condition_code,
        COUNT(*) AS demand
    FROM trips
    WHERE pickup_zone IS NOT NULL
      AND condition_code IS NOT NULL
    GROUP BY pickup_zone, condition_code
)

SELECT
    pickup_zone,
    ROUND(STDDEV(demand), 2) AS demand_variation,
    MIN(demand) AS min_demand,
    MAX(demand) AS max_demand,
    ROUND(AVG(demand),2) as avg_demand
FROM weather_demand
GROUP BY pickup_zone
ORDER BY demand_variation DESC
""").show(20, truncate=False)


+----------------------------+----------------+----------+----------+----------+
|pickup_zone                 |demand_variation|min_demand|max_demand|avg_demand|
+----------------------------+----------------+----------+----------+----------+
|Midtown Center              |45605.54        |632       |135673    |29886.0   |
|JFK Airport                 |44311.55        |531       |135391    |28909.43  |
|Upper East Side South       |43841.6         |688       |133384    |29329.5   |
|Upper East Side North       |40266.66        |707       |125038    |27330.14  |
|Midtown East                |33475.87        |420       |100356    |22394.5   |
|Times Sq/Theatre District   |32220.48        |505       |94730     |21554.07  |
|Penn Station/Madison Sq West|31699.27        |338       |97107     |21456.21  |
|LaGuardia Airport           |30882.6         |327       |96824     |19633.86  |
|Lincoln Square East         |30864.83        |361       |91086     |20700.79  |
|Midtown North              

In [ ]:
#q5
spark.sql("""
WITH hourly AS (
    SELECT
        DAYOFWEEK(pickup_datetime) AS day_of_week,
        DATE_FORMAT(pickup_datetime, 'EEEE') AS day_name,
        HOUR(pickup_datetime) AS hour_of_day,
        COUNT(*) AS trip_count
    FROM trips
    WHERE pickup_datetime IS NOT NULL
    GROUP BY
        DAYOFWEEK(pickup_datetime),
        DATE_FORMAT(pickup_datetime, 'EEEE'),
        HOUR(pickup_datetime)
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY day_of_week
            ORDER BY trip_count DESC
        ) AS rn
    FROM hourly
)
SELECT
    day_name,
    CONCAT(LPAD(CAST(hour_of_day AS STRING), 2, '0'), ':00') AS peak_hour,
    trip_count
FROM ranked
WHERE rn = 1
ORDER BY day_of_week
""").show()


+---------+---------+----------+
| day_name|peak_hour|trip_count|
+---------+---------+----------+
|   Sunday|    17:00|     69177|
|   Monday|    18:00|     75202|
|  Tuesday|    18:00|     90724|
|Wednesday|    18:00|     98863|
| Thursday|    18:00|    105093|
|   Friday|    18:00|     91754|
| Saturday|    18:00|     83820|
+---------+---------+----------+



In [30]:
#q6
spark.sql("""
WITH monthly AS (
    SELECT
        month,
        COUNT(*) AS taxi_demand
    FROM trips
    WHERE year = 2024
    GROUP BY month
),
with_previous AS (
    SELECT
        month,
        taxi_demand,
        LAG(taxi_demand) OVER (ORDER BY month) AS previous_month_demand
    FROM monthly
)
SELECT
    month,
    taxi_demand,
    ROUND(
        100.0 * (taxi_demand - previous_month_demand)
        / previous_month_demand,
        2
    ) AS mom_change_pct
FROM with_previous
ORDER BY month
""").show()



+-----+-----------+--------------+
|month|taxi_demand|mom_change_pct|
+-----+-----------+--------------+
|    1|    2724200|          NULL|
|    2|    2720031|         -0.15|
|    3|    3036585|         11.64|
|    4|          2|       -100.00|
+-----+-----------+--------------+



### w2 task 3

In [33]:
import time

def run_and_time(query):
    start = time.perf_counter()
    result = spark.sql(query).collect()
    elapsed = time.perf_counter() - start
    
    print(f"Execution time: {elapsed:.3f} seconds")
    return result, elapsed


In [31]:
cache_test_query = """
SELECT
    month,
    COUNT(*) AS taxi_demand
FROM trips
WHERE year = 2024
GROUP BY month
ORDER BY month
"""

In [32]:
# Making sure nothing is Cached
spark.catalog.clearCache()

print("Trips cached:", spark.catalog.isCached("trips"))


Trips cached: False


In [ ]:
original_result, original_time = run_and_time(cache_test_query) # Gave 1.308

Execution time: 1.308 seconds


In [35]:
spark.sql("CACHE TABLE trips")

DataFrame[]

In [36]:
spark.sql("SELECT COUNT(*) FROM trips").collect() #Materialize the cache
print("Trips cached:", spark.catalog.isCached("trips"))

Trips cached: True


In [37]:
cached_result, cached_time = run_and_time(cache_test_query)

Execution time: 0.659 seconds


In [38]:
cached_result_2, cached_time_2 = run_and_time(cache_test_query)

Execution time: 0.386 seconds


### Verify that caching didn't change the result

In [39]:
print("Results identical:", original_result == cached_result)

Results identical: True


### Look at the physical plan

In [46]:
spark.catalog.clearCache()
spark.sql("""
SELECT
    month,
    COUNT(*) AS taxi_demand
FROM trips
WHERE year = 2024
GROUP BY month
ORDER BY month
""").explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(year#67), (year#67 = 2024)]
ReadSchema: struct<>

(2) Project
Output [1]: [month#68]
Input [2]: [year#67, month#68]

(3) HashAggregate
Input [1]: [month#68]
Keys [1]: [month#68]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#14792L]
Results [2]: [month#68, count#14793L]

(4) Exchange
Input [2]: [month#68, count#14793L]
Arguments: hashpartitioning(month#68, 8), ENSURE_REQUIREMENTS, [plan_id=3670]

(5) HashAggregate
Input [2]: [month#68, count#14793L]
Keys [1]: [month#68]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [co

In [47]:
spark.sql("CACHE TABLE trips")
spark.sql("SELECT COUNT(*) FROM trips").collect()
spark.sql("""
SELECT
    month,
    COUNT(*) AS taxi_demand
FROM trips
WHERE year = 2024
GROUP BY month
ORDER BY month
""").explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (13)
+- Sort (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Exchange (9)
            +- HashAggregate (8)
               +- Project (7)
                  +- Filter (6)
                     +- Scan In-memory table trips (1)
                           +- InMemoryRelation (2)
                                 +- * Project (5)
                                    +- * ColumnarToRow (4)
                                       +- Scan parquet  (3)


(1) Scan In-memory table trips
Output [2]: [month#68, year#67]
Arguments: [month#68, year#67], [isnotnull(year#67), (year#67 = 2024)]

(2) InMemoryRelation
Arguments: [vendor_id#48, pickup_datetime#49, dropoff_datetime#50, passenger_count#51, trip_distance#52, rate_code_id#53, store_and_fwd_flag#54, pickup_location_id#55, dropoff_location_id#56, payment_type#57, fare_amount#58, extra#59, mta_tax#60, tip_amount#61, tolls_amount#62, improvement_surcharge#63, total_amount#64, congestion_surcharge#

The physical plan after caching contains an InMemoryRelation and Scan In-memory table trips, confirming that Spark reads the cached Trips table rather than directly scanning the underlying Delta/Parquet files.

### Improvement

In [48]:
improvement = ((original_time - cached_time) / original_time) * 100

print(f"Original: {original_time:.3f} seconds")
print(f"Cached:   {cached_time:.3f} seconds")
print(f"Improvement: {improvement:.2f}%")

Original: 1.308 seconds
Cached:   0.659 seconds
Improvement: 49.57%


Why was caching appropriate?

The Trips table is the central dataset used repeatedly by our analytical queries. Reading the same underlying Delta data repeatedly can add I/O overhead. Caching allows Spark to keep the table's data in memory for subsequent queries.

The trade-off is:

- Benefit: faster repeated queries.

- Cost: uses executor memory.

- When useful: frequently accessed data.

- When not useful: data queried only once, or data so large that caching causes memory pressure/eviction.



Conclusion: Caching the frequently accessed trips table reduced the execution time of the 2024 monthly taxi-demand query from 1.308 seconds to 0.659 seconds, representing a 49.57% reduction in measured execution time. The results before and after caching were identical. The physical execution plan changed from reading the underlying files to using InMemoryRelation and Scan In-memory table trips, confirming that the cache was utilized. The main trade-off is additional memory/disk usage, so caching is most beneficial when the same dataset is accessed repeatedly.

## Pruning

In [50]:
spark.sql("""
DESCRIBE DETAIL delta.`f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips`
""").select("partitionColumns").show(truncate=False)

+----------------+
|partitionColumns|
+----------------+
|[year, month]   |
+----------------+



In [51]:
# Clear cache
spark.catalog.clearCache()

# Baseline query
baseline_query = """
SELECT
    year,
    month,
    COUNT(*) AS taxi_demand
FROM delta.`f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips`
GROUP BY year, month
ORDER BY year, month
"""

In [52]:
# Run & Time it
import time

start = time.perf_counter()
baseline_result = spark.sql(baseline_query).collect()
baseline_time = time.perf_counter() - start
print(f"Baseline execution time: {baseline_time:.3f} seconds")

Baseline execution time: 0.715 seconds


In [53]:
# We are interested only in 2024:
optimized_query = """
SELECT
    month,
    COUNT(*) AS taxi_demand
FROM delta.`f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips`
WHERE year = 2024
GROUP BY month
ORDER BY month
"""
start = time.perf_counter()
optimized_result = spark.sql(optimized_query).collect()
optimized_time = time.perf_counter() - start

print(f"Optimized execution time: {optimized_time:.3f} seconds")

Optimized execution time: 0.587 seconds


In [54]:
improvement = ((baseline_time - optimized_time) / baseline_time) * 100

print(f"Performance improvement: {improvement:.2f}%")


Performance improvement: 17.93%


In [55]:
spark.sql(optimized_query).explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [year#18760, month#18761]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(year#18760), (year#18760 = 2024)]
ReadSchema: struct<>

(2) Project
Output [1]: [month#18761]
Input [2]: [year#18760, month#18761]

(3) HashAggregate
Input [1]: [month#18761]
Keys [1]: [month#18761]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#18889L]
Results [2]: [month#18761, count#18890L]

(4) Exchange
Input [2]: [month#18761, count#18890L]
Arguments: hashpartitioning(month#18761, 8), ENSURE_REQUIREMENTS, [plan_id=4251]

(5) HashAggregate
Input [2]: [month#18761, count#18890L]
Keys [1]: [month#18761]
Functions [1]

Focus on this part: PartitionFilters: [isnotnull(year#18760), (year#18760 = 2024)]

"This data is partitioned by year/month, and I only need year 2024, so don't bother reading the other year partitions." -Spark

In [56]:
#Verify results
# The baseline contains every year, while the optimized query contains only 2024.
# So extract the 2024 rows from the baseline:
baseline_2024 = [
    (row["month"], row["taxi_demand"])
    for row in baseline_result
    if row["year"] == 2024
]

optimized_2024 = [
    (row["month"], row["taxi_demand"])
    for row in optimized_result
]

print("Results identical:", baseline_2024 == optimized_2024)

Results identical: True


Partition pruning: The integrated taxi trips Delta table is partitioned by year and month. The optimized query restricted the analysis to year = 2024, allowing Spark to skip partitions belonging to other years. Execution time decreased from 0.715 seconds to 0.587 seconds, a 17.9% reduction. The physical plan confirmed partition pruning through PartitionFilters: [isnotnull(year), (year = 2024)]. The results were verified to be identical to the 2024 subset of the unfiltered query. The trade-off is that partition pruning only provides benefits when queries filter on partition columns; excessive or poorly chosen partitioning can also lead to many small files.

Why was partition pruning appropriate?
The Trips table is partitioned by year and month, and the selected query only analyzes data from 2024. Filtering with WHERE year = 2024 allows Spark to read only the relevant year partitions instead of scanning data from other years.

The trade-off is:

Benefit: reduces the amount of data Spark needs to read, improving query performance.

Cost: provides limited benefit when queries don't filter on partition columns.

When useful: queries that filter on year, month, or other partition columns.

When not useful: queries that need data from all partitions or filter mainly on non-partition columns.

## Broadacst Join

 idea is that instead of shuffling the huge Trips table around the cluster to perform the join, Spark can broadcast the small lookup table to the executors.

In [57]:
spark.sql("SHOW TABLES").show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |trips    |true       |
+---------+---------+-----------+



In [60]:
import os

gold_path = "f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold"

for item in os.listdir(gold_path):
    print(item)


integrated_taxi_trips
products


In [62]:
# Load the lookup
zone_lookup = spark.read.option("header", True).option("inferSchema", True).csv(
    "f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/raw/taxi_zones/taxi_zone_lookup.csv"
)

zone_lookup.printSchema()
zone_lookup.show(5)


root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [64]:
zone_lookup.createOrReplaceTempView("zone_lookup")
zone_lookup.count()

265

In [65]:
spark.catalog.clearCache()
baseline_join_query = """
SELECT
    z.Borough,
    COUNT(*) AS trip_count
FROM trips t
JOIN zone_lookup z
    ON t.pickup_location_id = z.LocationID
GROUP BY z.Borough
ORDER BY trip_count DESC
"""

import time

start = time.perf_counter()

baseline_join_result = spark.sql(baseline_join_query).collect()

baseline_join_time = time.perf_counter() - start

print(f"Baseline join time: {baseline_join_time:.3f} seconds")

Baseline join time: 2.204 seconds


In [66]:
spark.sql(baseline_join_query).explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- BroadcastHashJoin Inner BuildRight (7)
                     :- Project (3)
                     :  +- Filter (2)
                     :     +- Scan parquet  (1)
                     +- BroadcastExchange (6)
                        +- Filter (5)
                           +- Scan csv  (4)


(1) Scan parquet 
Output [3]: [pickup_location_id#55, year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(pickup_location_id)]
ReadSchema: struct<pickup_location_id:int>

(2) Filter
Input [3]: [pickup_location_id#55, year#67, month#68]
Condition : isnotnull(pickup_location_id#55)

(3) Project
Output [1]: [pickup_location_id#55]
Input [3]: [pickup_location_id

"baseline" is already optimized by Spark's automatic broadcast-join decision. so lets disable broadcast for now for unbiased result

In [67]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
start = time.perf_counter()

baseline_join_result = spark.sql(baseline_join_query).collect()

baseline_join_time = time.perf_counter() - start

print(f"Baseline join time: {baseline_join_time:.3f} seconds")
spark.sql(baseline_join_query).explain("formatted")

Baseline join time: 3.974 seconds
== Physical Plan ==
AdaptiveSparkPlan (17)
+- Sort (16)
   +- Exchange (15)
      +- HashAggregate (14)
         +- Exchange (13)
            +- HashAggregate (12)
               +- Project (11)
                  +- SortMergeJoin Inner (10)
                     :- Sort (5)
                     :  +- Exchange (4)
                     :     +- Project (3)
                     :        +- Filter (2)
                     :           +- Scan parquet  (1)
                     +- Sort (9)
                        +- Exchange (8)
                           +- Filter (7)
                              +- Scan csv  (6)


(1) Scan parquet 
Output [3]: [pickup_location_id#55, year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(pickup_location_id)]
ReadSchema: struct<pickup_location_id:int>

(2) Filter
Input [3]: [pickup_location_id#5

SortMergeJoin found. 

In [69]:
broadcast_join_query = """
SELECT /*+ BROADCAST(z) */
    z.Borough,
    COUNT(*) AS trip_count
FROM trips t
JOIN zone_lookup z
    ON t.pickup_location_id = z.LocationID
GROUP BY z.Borough
ORDER BY trip_count DESC
"""

start = time.perf_counter()

broadcast_join_result = spark.sql(broadcast_join_query).collect()

broadcast_join_time = time.perf_counter() - start

print(f"Broadcast join time: {broadcast_join_time:.3f} seconds")
spark.sql(broadcast_join_query).explain("formatted")

Broadcast join time: 1.198 seconds
== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- BroadcastHashJoin Inner BuildRight (7)
                     :- Project (3)
                     :  +- Filter (2)
                     :     +- Scan parquet  (1)
                     +- BroadcastExchange (6)
                        +- Filter (5)
                           +- Scan csv  (4)


(1) Scan parquet 
Output [3]: [pickup_location_id#55, year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(pickup_location_id)]
ReadSchema: struct<pickup_location_id:int>

(2) Filter
Input [3]: [pickup_location_id#55, year#67, month#68]
Condition : isnotnull(pickup_location_id#55)

(3) Project
Output [1]: [pickup_location_id

Imp ones here: (6) BroadcastExchange
Input [2]: [LocationID#18988, Borough#18989]
Arguments: HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=5378]

(7) BroadcastHashJoin
Left keys [1]: [pickup_location_id#55]
Right keys [1]: [LocationID#18988]
Join type: Inner
Join condition: None

In [70]:
improvement = (
    (baseline_join_time - broadcast_join_time)
    / baseline_join_time
) * 100

print(f"Performance improvement: {improvement:.2f}%")


Performance improvement: 69.86%


In [71]:
print("Results identical:",
      baseline_join_result == broadcast_join_result)


Results identical: True


Why was broadcast join appropriate?

The Taxi Zone Lookup table is very small compared with the Trips table. Broadcasting the lookup table allows Spark to distribute the small dataset to the executors, avoiding the need to shuffle the large Trips dataset for the join.

Trade-offs:

Benefit: reduces data shuffling and can significantly improve join performance.

Cost: the broadcast table must fit comfortably in executor memory.

When useful: joining a large dataset with a small lookup/reference dataset.

When not useful: joining two large tables, or when the table being broadcast is too large to fit safely in executor memory.


In our experiment, execution time decreased from 3.974 seconds to 1.198 seconds, representing a 69.86% reduction. The physical plan showed BroadcastExchange and BroadcastHashJoin, confirming that the broadcast optimization was applied.

## AQE
The goal is to let Spark dynamically modify the execution plan based on the actual data it encounters during execution.

In [72]:
aqe_query = """
SELECT
    z.Borough,
    COUNT(*) AS trip_count,
    AVG(t.trip_distance) AS avg_distance
FROM trips t
JOIN zone_lookup z
    ON t.pickup_location_id = z.LocationID
GROUP BY z.Borough
ORDER BY trip_count DESC
"""
#Enable Automatic Broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

In [73]:
# Disable AQE and verify
spark.conf.set("spark.sql.adaptive.enabled", "false")
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))

AQE: false


In [74]:
import time

start = time.perf_counter()

aqe_off_result = spark.sql(aqe_query).collect()

aqe_off_time = time.perf_counter() - start

print(f"AQE disabled: {aqe_off_time:.3f} seconds")


AQE disabled: 2.253 seconds


In [75]:
spark.sql(aqe_query).explain("formatted")

== Physical Plan ==
* Sort (14)
+- Exchange (13)
   +- * HashAggregate (12)
      +- Exchange (11)
         +- * HashAggregate (10)
            +- * Project (9)
               +- * BroadcastHashJoin Inner BuildRight (8)
                  :- * Project (4)
                  :  +- * Filter (3)
                  :     +- * ColumnarToRow (2)
                  :        +- Scan parquet  (1)
                  +- BroadcastExchange (7)
                     +- * Filter (6)
                        +- Scan csv  (5)


(1) Scan parquet 
Output [4]: [trip_distance#52, pickup_location_id#55, year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(pickup_location_id)]
ReadSchema: struct<trip_distance:double,pickup_location_id:int>

(2) ColumnarToRow [codegen id : 2]
Input [4]: [trip_distance#52, pickup_location_id#55, year#67, month#68]

(3) Filter [codegen id : 2]
Input [4]

Normal.

In [76]:
# Enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))

AQE: true


In [77]:
# Runnnning same query
start = time.perf_counter()

aqe_on_result = spark.sql(aqe_query).collect()

aqe_on_time = time.perf_counter() - start

print(f"AQE enabled: {aqe_on_time:.3f} seconds")


AQE enabled: 1.315 seconds


In [78]:
improvement = (
    (aqe_off_time - aqe_on_time)
    / aqe_off_time
) * 100

print(f"AQE performance improvement: {improvement:.2f}%")


AQE performance improvement: 41.62%


In [79]:
print("Results identical:",
      aqe_off_result == aqe_on_result)


Results identical: True


In [80]:
spark.sql(aqe_query).explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- BroadcastHashJoin Inner BuildRight (7)
                     :- Project (3)
                     :  +- Filter (2)
                     :     +- Scan parquet  (1)
                     +- BroadcastExchange (6)
                        +- Filter (5)
                           +- Scan csv  (4)


(1) Scan parquet 
Output [4]: [trip_distance#52, pickup_location_id#55, year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(pickup_location_id)]
ReadSchema: struct<trip_distance:double,pickup_location_id:int>

(2) Filter
Input [4]: [trip_distance#52, pickup_location_id#55, year#67, month#68]
Condition : isnotnull(pickup_location_id#55)

(3) Project
Output [2

(14) AdaptiveSparkPlan found

Why was AQE appropriate?
Adaptive Query Execution was appropriate because the selected query contains joins, aggregations, and shuffle operations. AQE allows Spark to use runtime statistics collected during execution to adapt the physical execution plan and potentially reduce unnecessary shuffle overhead or improve join execution.

Trade-offs
Benefit: Spark can adapt execution decisions based on the actual runtime data.

Cost: AQE introduces some runtime planning overhead.

When useful: complex queries involving joins, aggregations, and shuffles where actual data characteristics may differ from estimates.

When less useful: very simple queries with little or no shuffle where there are few opportunities for adaptive optimization.

Your measured result
With AQE disabled, the query took 2.253 seconds. With AQE enabled, execution time decreased to 1.315 seconds, representing a 41.63% reduction. The results were identical in both cases.

Caching: InMemoryRelation, Scan In-memory table

Partition pruning: PartitionFilters: ... year = 2024

Broadcast: BroadcastExchange, BroadcastHashJoin

AQE: AdaptiveSparkPlan

In [3]:
# Helper: run a query N times, return median seconds
import statistics

def bench(spark, sql, n=3, label=''):
    times = []
    for i in range(n):
        start = time.time()
        spark.sql(sql).collect()
        times.append(round(time.time() - start, 2))
    med = round(statistics.median(times), 2)
    print(f'{label}: runs={times}  median={med}s')
    return med

# Helper: verify two SQL queries produce identical results
def verify_same(spark, sql1, sql2):
    df1 = spark.sql(sql1)
    df2 = spark.sql(sql2)
    diff = df1.subtract(df2).count() + df2.subtract(df1).count()
    print('Results identical:', diff == 0)

results = {}  # store {label: seconds} for summary table

In [5]:
q1_sql = """
    SELECT year, month, pickup_location_id, pickup_zone, COUNT(*) AS trip_count
    FROM {view}
    WHERE year >= 2024
    GROUP BY year, month, pickup_location_id, pickup_zone
    ORDER BY year, month, trip_count DESC
"""

# Baseline – no cache
results['Q1 baseline'] = bench(spark, q1_sql.format(view='trips'), label='Q1 no cache')

Q1 no cache: runs=[7.4, 1.54, 1.24]  median=1.54s


In [6]:
# Cache the filtered trips
cached = spark.sql('SELECT * FROM trips WHERE year >= 2024').cache()
cached.createOrReplaceTempView('trips_cached')
cached.count()  # materialise
print('Cache materialised')

results['Q1 cached'] = bench(spark, q1_sql.format(view='trips_cached'), label='Q1 cached')

# Verify same results
verify_same(spark, q1_sql.format(view='trips'), q1_sql.format(view='trips_cached'))

Cache materialised
Q1 cached: runs=[1.06, 0.84, 0.81]  median=0.84s
Results identical: True


In [7]:
q6_no_prune = """
    SELECT year, month, COUNT(*) AS trip_count
    FROM trips
    GROUP BY year, month
    ORDER BY year, month
"""

q6_pruned = """
    SELECT year, month, COUNT(*) AS trip_count
    FROM trips
    WHERE year = 2024
    GROUP BY year, month
    ORDER BY year, month
"""

results['Q6 no prune'] = bench(spark, q6_no_prune, label='Q6 no pruning')
results['Q6 pruned']   = bench(spark, q6_pruned,   label='Q6 pruned')

# Check EXPLAIN for PartitionFilters
print('\n--- EXPLAIN no prune ---')
spark.sql(q6_no_prune).explain('formatted')
print('\n--- EXPLAIN pruned ---')
spark.sql(q6_pruned).explain('formatted')

Q6 no pruning: runs=[1.28, 0.63, 0.57]  median=0.63s
Q6 pruned: runs=[0.73, 0.5, 0.5]  median=0.5s

--- EXPLAIN no prune ---
== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Exchange (3)
            +- HashAggregate (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
ReadSchema: struct<>

(2) HashAggregate
Input [2]: [year#67, month#68]
Keys [2]: [year#67, month#68]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#8336L]
Results [3]: [year#67, month#68, count#8337L]

(3) Exchange
Input [3]: [year#67, month#68, count#8337L]
Arguments: hashpartitioning(year#67, month#68, 8), ENSURE_REQUIREMENTS, [plan_id=2464]

(4) HashAggregate
Input [3]: [year#67, month#68, count#8337L]
Keys [2]: [year#67, month#68]
Functions [1]: [count(1)]
Ag

In [8]:
q2_no_hint = """
    SELECT t.condition_code, cl.condition_name,
           ROUND(AVG(trip_distance), 3) AS avg_dist, COUNT(*) AS trip_count
    FROM trips t
    LEFT JOIN condition_labels cl ON t.condition_code = cl.condition_code
    WHERE trip_distance > 0
    GROUP BY t.condition_code, cl.condition_name
"""

q2_broadcast = """
    SELECT /*+ BROADCAST(cl) */ t.condition_code, cl.condition_name,
           ROUND(AVG(trip_distance), 3) AS avg_dist, COUNT(*) AS trip_count
    FROM trips t
    LEFT JOIN condition_labels cl ON t.condition_code = cl.condition_code
    WHERE trip_distance > 0
    GROUP BY t.condition_code, cl.condition_name
"""

results['Q2 no hint']   = bench(spark, q2_no_hint,   label='Q2 no broadcast')
results['Q2 broadcast'] = bench(spark, q2_broadcast, label='Q2 broadcast')

verify_same(spark, q2_no_hint, q2_broadcast)

print('\n--- EXPLAIN no hint ---')
spark.sql(q2_no_hint).explain('formatted')
print('\n--- EXPLAIN broadcast ---')
spark.sql(q2_broadcast).explain('formatted')

Q2 no broadcast: runs=[2.16, 0.82, 0.8]  median=0.82s
Q2 broadcast: runs=[0.87, 0.7, 0.76]  median=0.76s
Results identical: True

--- EXPLAIN no hint ---
== Physical Plan ==
AdaptiveSparkPlan (11)
+- HashAggregate (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- Project (7)
            +- BroadcastHashJoin LeftOuter BuildRight (6)
               :- Project (3)
               :  +- Filter (2)
               :     +- Scan parquet  (1)
               +- BroadcastExchange (5)
                  +- LocalTableScan (4)


(1) Scan parquet 
Output [4]: [trip_distance#52, condition_code#76, year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(trip_distance), GreaterThan(trip_distance,0.0)]
ReadSchema: struct<trip_distance:double,condition_code:int>

(2) Filter
Input [4]: [trip_distance#52, condition_code#76, year#67, month#68]
Condition : (isnotnull(t

In [9]:
# Force sort-merge join by disabling auto broadcast
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
results['Q2 no broadcast forced'] = bench(spark, q2_no_hint, label='Q2 forced sort-merge')

# Re-enable
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '104857600')
results['Q2 broadcast auto'] = bench(spark, q2_no_hint, label='Q2 auto broadcast')

Q2 forced sort-merge: runs=[4.8, 3.02, 2.96]  median=3.02s
Q2 auto broadcast: runs=[0.68, 0.78, 0.65]  median=0.68s


In [10]:
q3_sql = """
    SELECT year, month,
        CASE
            WHEN pm25_hourly_avg <= 12  THEN 'Good'
            WHEN pm25_hourly_avg <= 35  THEN 'Moderate'
            WHEN pm25_hourly_avg <= 55  THEN 'Unhealthy for Sensitive Groups'
            ELSE 'Unhealthy'
        END AS aqi_category,
        COUNT(*) AS trip_count,
        ROUND(AVG(pm25_hourly_avg), 1) AS avg_pm25
    FROM trips
    WHERE pm25_hourly_avg IS NOT NULL
    GROUP BY year, month, aqi_category
    ORDER BY year, month
"""

spark.conf.set('spark.sql.adaptive.enabled', 'false')
results['Q3 AQE off'] = bench(spark, q3_sql, label='Q3 AQE off')

spark.conf.set('spark.sql.adaptive.enabled', 'true')
results['Q3 AQE on']  = bench(spark, q3_sql, label='Q3 AQE on')

print('\n--- EXPLAIN AQE off ---')
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.sql(q3_sql).explain('formatted')

print('\n--- EXPLAIN AQE on ---')
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.sql(q3_sql).explain('formatted')

Q3 AQE off: runs=[1.16, 0.71, 0.6]  median=0.71s
Q3 AQE on: runs=[0.66, 0.55, 0.64]  median=0.64s

--- EXPLAIN AQE off ---
== Physical Plan ==
* Sort (9)
+- Exchange (8)
   +- * HashAggregate (7)
      +- Exchange (6)
         +- * HashAggregate (5)
            +- * Project (4)
               +- * Filter (3)
                  +- * ColumnarToRow (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pm25_hourly_avg#77, year#67, month#68]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PushedFilters: [IsNotNull(pm25_hourly_avg)]
ReadSchema: struct<pm25_hourly_avg:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [pm25_hourly_avg#77, year#67, month#68]

(3) Filter [codegen id : 1]
Input [3]: [pm25_hourly_avg#77, year#67, month#68]
Condition : isnotnull(pm25_hourly_avg#77)

(4) Project [codegen id : 1]
Output [4]: [year#67, month#68, pm25_hourly_avg#77, CASE WHEN (pm25

In [11]:
print(f'{'Experiment':<25} {'Time (s)':>10}')
print('-' * 37)
for label, t in results.items():
    print(f'{label:<25} {t:>10.2f}')

# Speedups
print()
pairs = [
    ('Q1 baseline', 'Q1 cached',    'Caching'),
    ('Q6 no prune', 'Q6 pruned',    'Partition pruning'),
    ('Q2 no hint',  'Q2 broadcast', 'Broadcast join'),
    ('Q3 AQE off',  'Q3 AQE on',   'AQE'),
]
for base_key, opt_key, name in pairs:
    if base_key in results and opt_key in results:
        speedup = round(results[base_key] / results[opt_key], 2)
        print(f'{name}: {results[base_key]}s -> {results[opt_key]}s  ({speedup}x speedup)')

Experiment                  Time (s)
-------------------------------------
Q1 baseline                     1.54
Q1 cached                       0.84
Q6 no prune                     0.63
Q6 pruned                       0.50
Q2 no hint                      0.82
Q2 broadcast                    0.76
Q2 no broadcast forced          3.02
Q2 broadcast auto               0.68
Q3 AQE off                      0.71
Q3 AQE on                       0.64

Caching: 1.54s -> 0.84s  (1.83x speedup)
Partition pruning: 0.63s -> 0.5s  (1.26x speedup)
Broadcast join: 0.82s -> 0.76s  (1.08x speedup)
AQE: 0.71s -> 0.64s  (1.11x speedup)


T4

In [26]:
"""
Task 4 – Reusable Analytical Data Products
==========================================
Generates four Delta tables from the integrated taxi trip dataset:

  1. daily_mobility_summary       – daily trip counts / distances per zone
  2. weather_impact_summary       – avg distance & demand per weather bucket
  3. air_quality_impact_summary   – demand & avg PM2.5 per AQI category
  4. taxi_zone_statistics         – per-zone demand variance across weather

Each table includes metadata columns:
    _data_source      – originating view / table
    _schema_version   – semantic version string
    _created_at       – first time this product was generated (UTC)
    _refreshed_at     – current run timestamp (UTC)
"""

'\nTask 4 – Reusable Analytical Data Products\n==========================================\nGenerates four Delta tables from the integrated taxi trip dataset:\n\n  1. daily_mobility_summary       – daily trip counts / distances per zone\n  2. weather_impact_summary       – avg distance & demand per weather bucket\n  3. air_quality_impact_summary   – demand & avg PM2.5 per AQI category\n  4. taxi_zone_statistics         – per-zone demand variance across weather\n\nEach table includes metadata columns:\n    _data_source      – originating view / table\n    _schema_version   – semantic version string\n    _created_at       – first time this product was generated (UTC)\n    _refreshed_at     – current run timestamp (UTC)\n'

### Task 4

In [3]:
print("=== INTEGRATED DATASET SCHEMA ===")
trips.printSchema()

print("\n=== ROW COUNT ===")
print(trips.count())

print("\n=== COLUMNS ===")
print(trips.columns)

=== INTEGRATED DATASET SCHEMA ===
root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- picku

In [4]:
print("=== SAMPLE ===")
trips.show(5, truncate=False)


=== SAMPLE ===
+---------+-------------------+-------------------+---------------+-------------+------------+------------------+------------------+-------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----+-----+-------------------+----+------------+-------------+----------+--------------+--------+--------------+---------------+-------------------+--------------+-------------------+-------------------+---------------+--------------------+
|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|rate_code_id|store_and_fwd_flag|pickup_location_id|dropoff_location_id|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|year|month|pickup_hour        |temp|rel_humidity|precipitation|wind_speed|wind_direction|pressure|condition_code|pm25_hourly_avg|pickup_zone        |pickup_borough|pickup_servic

In [5]:
from delta.tables import DeltaTable

integrated_path = f"{BASE}/gold/integrated_taxi_trips"

detail = (
    spark.sql(f"""
        DESCRIBE DETAIL delta.`{integrated_path}`
    """)
)

detail.select(
    "format",
    "numFiles",
    "sizeInBytes",
    "partitionColumns"
).show(truncate=False)


+------+--------+-----------+----------------+
|format|numFiles|sizeInBytes|partitionColumns|
+------+--------+-----------+----------------+
|delta |15      |212196141  |[year, month]   |
+------+--------+-----------+----------------+



In [6]:
from pyspark.sql import functions as F

print("=== YEAR / MONTH COVERAGE ===")

trips.groupBy("year", "month") \
    .count() \
    .orderBy("year", "month") \
    .show(50)

print("=== IMPORTANT COLUMN COMPLETENESS ===")

trips.select(
    F.count("*").alias("total_rows"),
    F.count("pickup_datetime").alias("pickup_datetime"),
    F.count("trip_distance").alias("trip_distance"),
    F.count("total_amount").alias("total_amount"),
    F.count("condition_code").alias("condition_code"),
    F.count("temp").alias("temp"),
    F.count("pm25_hourly_avg").alias("pm25_hourly_avg"),
    F.count("pickup_zone").alias("pickup_zone"),
    F.count("pickup_borough").alias("pickup_borough")
).show()


=== YEAR / MONTH COVERAGE ===
+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2002|   12|      3|
|2008|   12|      1|
|2009|    1|      4|
|2023|   12|     10|
|2024|    1|2724200|
|2024|    2|2720031|
|2024|    3|3036585|
|2024|    4|      2|
+----+-----+-------+

=== IMPORTANT COLUMN COMPLETENESS ===
+----------+---------------+-------------+------------+--------------+-------+---------------+-----------+--------------+
|total_rows|pickup_datetime|trip_distance|total_amount|condition_code|   temp|pm25_hourly_avg|pickup_zone|pickup_borough|
+----------+---------------+-------------+------------+--------------+-------+---------------+-----------+--------------+
|   8480836|        8480836|      8480836|     8480836|       8480818|8480818|        8440808|    8480836|       8480836|
+----------+---------------+-------------+------------+--------------+-------+---------------+-----------+--------------+



In [7]:
print("=== WEATHER CONDITION CODES ===")

trips.groupBy("condition_code") \
    .count() \
    .orderBy("condition_code") \
    .show(30)


=== WEATHER CONDITION CODES ===
+--------------+-------+
|condition_code|  count|
+--------------+-------+
|          NULL|     18|
|             1| 207482|
|             2|2742193|
|             3|2002764|
|             4|1922398|
|             5| 244105|
|             7| 628442|
|             8| 208650|
|             9| 260892|
|            10|  12271|
|            12|  18343|
|            13|  15881|
|            14| 167467|
|            15|  29851|
|            16|  20079|
+--------------+-------+



In [8]:
print("=== BOROUGHS ===")

trips.groupBy("pickup_borough") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30)


=== BOROUGHS ===
+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|7610873|
|        Queens| 767108|
|      Brooklyn|  58120|
|       Unknown|  27414|
|         Bronx|  15526|
|           N/A|   1510|
|           EWR|    181|
| Staten Island|    104|
+--------------+-------+



1. Daily trip demand and mobility statistics by pickup zone.

In [9]:
print("=== DAILY MOBILITY CHECK ===")

trips.select(
    F.to_date("pickup_datetime").alias("trip_date")
).distinct().orderBy("trip_date").show(50)

print("Distinct dates:",
      trips.select(F.to_date("pickup_datetime").alias("trip_date")).distinct().count())

print("Distinct pickup zones:",
      trips.select("pickup_location_id").distinct().count())

print("Distinct pickup boroughs:",
      trips.select("pickup_borough").distinct().count())


=== DAILY MOBILITY CHECK ===
+----------+
| trip_date|
+----------+
|2002-12-31|
|2008-12-31|
|2009-01-01|
|2023-12-31|
|2024-01-01|
|2024-01-02|
|2024-01-03|
|2024-01-04|
|2024-01-05|
|2024-01-06|
|2024-01-07|
|2024-01-08|
|2024-01-09|
|2024-01-10|
|2024-01-11|
|2024-01-12|
|2024-01-13|
|2024-01-14|
|2024-01-15|
|2024-01-16|
|2024-01-17|
|2024-01-18|
|2024-01-19|
|2024-01-20|
|2024-01-21|
|2024-01-22|
|2024-01-23|
|2024-01-24|
|2024-01-25|
|2024-01-26|
|2024-01-27|
|2024-01-28|
|2024-01-29|
|2024-01-30|
|2024-01-31|
|2024-02-01|
|2024-02-02|
|2024-02-03|
|2024-02-04|
|2024-02-05|
|2024-02-06|
|2024-02-07|
|2024-02-08|
|2024-02-09|
|2024-02-10|
|2024-02-11|
|2024-02-12|
|2024-02-13|
|2024-02-14|
|2024-02-15|
+----------+
only showing top 50 rows

Distinct dates: 96
Distinct pickup zones: 258
Distinct pickup boroughs: 8


In [10]:
print("=== DAILY MOBILITY SAMPLE ===")

daily_test = (
    trips
    .filter(F.col("pickup_datetime").isNotNull())
    .groupBy(
        F.to_date("pickup_datetime").alias("trip_date"),
        "pickup_location_id",
        "pickup_zone",
        "pickup_borough"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("trip_distance").alias("total_distance"),
        F.avg("trip_distance").alias("avg_distance"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_fare")
    )
)

daily_test.show(10, truncate=False)


=== DAILY MOBILITY SAMPLE ===
+----------+------------------+-----------------------------+--------------+-----------+------------------+------------------+------------------+------------------+
|trip_date |pickup_location_id|pickup_zone                  |pickup_borough|total_trips|total_distance    |avg_distance      |total_revenue     |avg_fare          |
+----------+------------------+-----------------------------+--------------+-----------+------------------+------------------+------------------+------------------+
|2024-01-13|249               |West Village                 |Manhattan     |3489       |7188.339999999989 |2.06028661507595  |77156.09999999989 |22.114101461736855|
|2024-01-24|90                |Flatiron                     |Manhattan     |1489       |3090.0900000000024|2.0752787105439907|33857.35999999992 |22.73832102081929 |
|2024-01-13|132               |JFK Airport                  |Queens        |4400       |70631.69999999998 |16.05265909090909 |358234.7400000042 |

In [11]:
# ============================================================
# TASK 4 – PRODUCT 1 VALIDATION
# Daily Mobility Summary
# ============================================================

daily_mobility = spark.sql("""
    SELECT
        CAST(pickup_datetime AS DATE) AS trip_date,
        year,
        month,
        pickup_location_id,
        pickup_zone,
        pickup_borough,

        COUNT(*) AS total_trips,
        SUM(trip_distance) AS total_distance,
        AVG(trip_distance) AS avg_distance,
        SUM(total_amount) AS total_revenue,
        AVG(total_amount) AS avg_fare

    FROM trips

    WHERE pickup_datetime IS NOT NULL

    GROUP BY
        CAST(pickup_datetime AS DATE),
        year,
        month,
        pickup_location_id,
        pickup_zone,
        pickup_borough
""")

print("=== DAILY MOBILITY SUMMARY SCHEMA ===")
daily_mobility.printSchema()

print("\n=== ROW COUNT ===")
print("Product rows:", daily_mobility.count())

print("\n=== NULL CHECK ===")
daily_mobility.select(
    F.sum(F.col("trip_date").isNull().cast("int")).alias("null_dates"),
    F.sum(F.col("pickup_location_id").isNull().cast("int")).alias("null_zone_ids"),
    F.sum(F.col("pickup_zone").isNull().cast("int")).alias("null_zones"),
    F.sum(F.col("pickup_borough").isNull().cast("int")).alias("null_boroughs")
).show()

print("\n=== DATE RANGE ===")
daily_mobility.select(
    F.min("trip_date").alias("min_date"),
    F.max("trip_date").alias("max_date")
).show()

print("\n=== DAILY TOTALS ===")
daily_mobility.groupBy("trip_date") \
    .agg(F.sum("total_trips").alias("total_trips")) \
    .orderBy("trip_date") \
    .show(20)

print("\n=== SAMPLE ===")
daily_mobility.orderBy(
    F.col("trip_date").desc(),
    F.col("total_trips").desc()
).show(10, truncate=False)


=== DAILY MOBILITY SUMMARY SCHEMA ===
root
 |-- trip_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- total_trips: long (nullable = false)
 |-- total_distance: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_fare: double (nullable = true)


=== ROW COUNT ===
Product rows: 19793

=== NULL CHECK ===
+----------+-------------+----------+-------------+
|null_dates|null_zone_ids|null_zones|null_boroughs|
+----------+-------------+----------+-------------+
|         0|            0|         0|            0|
+----------+-------------+----------+-------------+


=== DATE RANGE ===
+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2002-12-31|2024-04-01|
+----------+----------+


=== DAILY TOTALS ===
+-

In [13]:
# ============================================================
# PRODUCT 2 – WEATHER DATA VALIDATION
# ============================================================

print("=== TEMPERATURE STATISTICS ===")

trips.select(
    F.count("temp").alias("non_null"),
    F.min("temp").alias("min_temp"),
    F.max("temp").alias("max_temp"),
    F.round(F.avg("temp"), 2).alias("avg_temp")
).show()

print("=== TEMPERATURE BUCKET COUNTS ===")

trips.select(
    F.when(F.col("temp") < 5, "Cold (<5°C)")
     .when(F.col("temp") < 20, "Mild (5–20°C)")
     .otherwise("Hot (>=20°C)")
     .alias("temp_bucket")
).groupBy("temp_bucket") \
 .count() \
 .orderBy("temp_bucket") \
 .show()

print("=== WEATHER CONDITIONS ===")

trips.groupBy("condition_code") \
    .count() \
    .orderBy("condition_code") \
    .show()

print("=== PRECIPITATION STATISTICS ===")

trips.select(
    F.count("precipitation").alias("non_null"),
    F.min("precipitation").alias("min_precipitation"),
    F.max("precipitation").alias("max_precipitation"),
    F.round(F.avg("precipitation"), 3).alias("avg_precipitation")
).show()

print("=== WIND SPEED STATISTICS ===")

trips.select(
    F.count("wind_speed").alias("non_null"),
    F.min("wind_speed").alias("min_wind_speed"),
    F.max("wind_speed").alias("max_wind_speed"),
    F.round(F.avg("wind_speed"), 2).alias("avg_wind_speed")
).show()


=== TEMPERATURE STATISTICS ===
+--------+--------+--------+--------+
|non_null|min_temp|max_temp|avg_temp|
+--------+--------+--------+--------+
| 8480818|    -7.2|    21.7|    5.32|
+--------+--------+--------+--------+

=== TEMPERATURE BUCKET COUNTS ===
+-------------+-------+
|  temp_bucket|  count|
+-------------+-------+
|  Cold (<5°C)|3793042|
| Hot (>=20°C)|  66362|
|Mild (5–20°C)|4621432|
+-------------+-------+

=== WEATHER CONDITIONS ===
+--------------+-------+
|condition_code|  count|
+--------------+-------+
|          NULL|     18|
|             1| 207482|
|             2|2742193|
|             3|2002764|
|             4|1922398|
|             5| 244105|
|             7| 628442|
|             8| 208650|
|             9| 260892|
|            10|  12271|
|            12|  18343|
|            13|  15881|
|            14| 167467|
|            15|  29851|
|            16|  20079|
+--------------+-------+

=== PRECIPITATION STATISTICS ===
+--------+-----------------+-----------

In [14]:
# ============================================================
# PRODUCT 2 – WEATHER IMPACT SUMMARY
# VALIDATION ONLY
# ============================================================

weather_impact = spark.sql("""
    SELECT
        CASE
            WHEN temp < 5 THEN 'Cold (<5°C)'
            WHEN temp < 20 THEN 'Mild (5–20°C)'
            ELSE 'Hot (>=20°C)'
        END AS temp_bucket,

        CASE condition_code
            WHEN 1  THEN 'Clear'
            WHEN 2  THEN 'Fair'
            WHEN 3  THEN 'Cloudy'
            WHEN 4  THEN 'Overcast'
            WHEN 5  THEN 'Fog'
            WHEN 6  THEN 'Freezing Fog'
            WHEN 7  THEN 'Light Rain'
            WHEN 8  THEN 'Rain'
            WHEN 9  THEN 'Heavy Rain'
            WHEN 10 THEN 'Freezing Rain'
            WHEN 11 THEN 'Heavy Freezing Rain'
            WHEN 12 THEN 'Sleet'
            WHEN 13 THEN 'Heavy Sleet'
            WHEN 14 THEN 'Light Snowfall'
            WHEN 15 THEN 'Snowfall'
            WHEN 16 THEN 'Heavy Snowfall'
            ELSE 'Unknown'
        END AS weather_condition,

        COUNT(*) AS total_trips,

        ROUND(AVG(temp), 2) AS avg_temperature,

        ROUND(AVG(precipitation), 3) AS avg_precipitation,

        ROUND(AVG(wind_speed), 2) AS avg_wind_speed,

        ROUND(AVG(trip_distance), 3) AS avg_distance_km,

        ROUND(AVG(total_amount), 2) AS avg_fare

    FROM trips

    WHERE temp IS NOT NULL
      AND condition_code IS NOT NULL
      AND trip_distance > 0

    GROUP BY
        temp_bucket,
        weather_condition

    ORDER BY
        temp_bucket,
        total_trips DESC
""")

print("=== WEATHER PRODUCT SCHEMA ===")
weather_impact.printSchema()

print("\n=== ROW COUNT ===")
print("Product rows:", weather_impact.count())

print("\n=== PRODUCT CONTENT ===")
weather_impact.show(50, truncate=False)


=== WEATHER PRODUCT SCHEMA ===
root
 |-- temp_bucket: string (nullable = false)
 |-- weather_condition: string (nullable = false)
 |-- total_trips: long (nullable = false)
 |-- avg_temperature: double (nullable = true)
 |-- avg_precipitation: double (nullable = true)
 |-- avg_wind_speed: double (nullable = true)
 |-- avg_distance_km: double (nullable = true)
 |-- avg_fare: double (nullable = true)


=== ROW COUNT ===
Product rows: 25

=== PRODUCT CONTENT ===
+-------------+-----------------+-----------+---------------+-----------------+--------------+---------------+--------+
|temp_bucket  |weather_condition|total_trips|avg_temperature|avg_precipitation|avg_wind_speed|avg_distance_km|avg_fare|
+-------------+-----------------+-----------+---------------+-----------------+--------------+---------------+--------+
|Cold (<5°C)  |Fair             |1636589    |0.86           |0.0              |22.0          |3.572          |27.7    |
|Cold (<5°C)  |Overcast         |802877     |1.43        

In [15]:
# ============================================================
# PRODUCT 2 – SOURCE RECONCILIATION
# ============================================================

source_weather_count = trips.filter(
    (F.col("temp").isNotNull()) &
    (F.col("condition_code").isNotNull()) &
    (F.col("trip_distance") > 0)
).count()

product_weather_count = weather_impact.agg(
    F.sum("total_trips").alias("trip_count")
).first()["trip_count"]

print("=== WEATHER PRODUCT RECONCILIATION ===")
print(f"Source qualifying trips : {source_weather_count:,}")
print(f"Product total trips     : {product_weather_count:,}")
print(f"Match                   : {source_weather_count == product_weather_count}")

print("\n=== WEATHER PRODUCT NULL CHECK ===")

weather_impact.select(
    F.sum(F.col("temp_bucket").isNull().cast("int")).alias("null_temp_bucket"),
    F.sum(F.col("weather_condition").isNull().cast("int")).alias("null_condition"),
    F.sum(F.col("avg_temperature").isNull().cast("int")).alias("null_temperature"),
    F.sum(F.col("avg_distance_km").isNull().cast("int")).alias("null_distance"),
    F.sum(F.col("avg_fare").isNull().cast("int")).alias("null_fare")
).show()


=== WEATHER PRODUCT RECONCILIATION ===
Source qualifying trips : 8,480,818
Product total trips     : 8,480,818
Match                   : True

=== WEATHER PRODUCT NULL CHECK ===
+----------------+--------------+----------------+-------------+---------+
|null_temp_bucket|null_condition|null_temperature|null_distance|null_fare|
+----------------+--------------+----------------+-------------+---------+
|               0|             0|               0|            0|        0|
+----------------+--------------+----------------+-------------+---------+



In [16]:
# ============================================================
# PRODUCT 3 – AIR QUALITY DATA VALIDATION
# ============================================================

print("=== PM2.5 STATISTICS ===")

trips.select(
    F.count("pm25_hourly_avg").alias("non_null"),
    F.min("pm25_hourly_avg").alias("min_pm25"),
    F.max("pm25_hourly_avg").alias("max_pm25"),
    F.round(F.avg("pm25_hourly_avg"), 2).alias("avg_pm25")
).show()

print("=== PM2.5 DISTRIBUTION ===")

trips.select(
    F.when(F.col("pm25_hourly_avg") <= 12, "Good")
     .when(F.col("pm25_hourly_avg") <= 35, "Moderate")
     .when(F.col("pm25_hourly_avg") <= 55, "Unhealthy for Sensitive Groups")
     .when(F.col("pm25_hourly_avg") <= 150, "Unhealthy")
     .otherwise("Very Unhealthy")
     .alias("aqi_category")
).groupBy("aqi_category") \
 .count() \
 .orderBy("aqi_category") \
 .show()

print("=== PM2.5 BY YEAR / MONTH ===")

trips.groupBy("year", "month") \
    .agg(
        F.count("pm25_hourly_avg").alias("trip_count"),
        F.round(F.avg("pm25_hourly_avg"), 2).alias("avg_pm25"),
        F.round(F.min("pm25_hourly_avg"), 2).alias("min_pm25"),
        F.round(F.max("pm25_hourly_avg"), 2).alias("max_pm25")
    ) \
    .orderBy("year", "month") \
    .show(30)


=== PM2.5 STATISTICS ===
+--------+--------+--------+--------+
|non_null|min_pm25|max_pm25|avg_pm25|
+--------+--------+--------+--------+
| 8440808|     0.7|    45.6|     8.5|
+--------+--------+--------+--------+

=== PM2.5 DISTRIBUTION ===
+--------------------+-------+
|        aqi_category|  count|
+--------------------+-------+
|                Good|6746309|
|            Moderate|1620225|
|Unhealthy for Sen...|  74274|
|      Very Unhealthy|  40028|
+--------------------+-------+

=== PM2.5 BY YEAR / MONTH ===
+----+-----+----------+--------+--------+--------+
|year|month|trip_count|avg_pm25|min_pm25|max_pm25|
+----+-----+----------+--------+--------+--------+
|2002|   12|         0|    NULL|    NULL|    NULL|
|2008|   12|         0|    NULL|    NULL|    NULL|
|2009|    1|         0|    NULL|    NULL|    NULL|
|2023|   12|         0|    NULL|    NULL|    NULL|
|2024|    1|   2696637|    7.83|     1.0|    31.4|
|2024|    2|   2713394|    9.84|     1.5|    45.6|
|2024|    3|   3030

In [21]:
from pyspark.sql import functions as F

pm25_dist = (
    trips
    .filter(F.col("pm25_hourly_avg").isNotNull())
    .withColumn(
        "aqi_category",
        F.when(F.col("pm25_hourly_avg") <= 12.0, "Good")
         .when(F.col("pm25_hourly_avg") <= 35.4, "Moderate")
         .when(F.col("pm25_hourly_avg") <= 55.4, "Unhealthy for Sensitive Groups")
         .otherwise("Very Unhealthy")
    )
    .groupBy("aqi_category")
    .count()
    .orderBy("aqi_category")
)

pm25_dist.show()


+--------------------+-------+
|        aqi_category|  count|
+--------------------+-------+
|                Good|6746309|
|            Moderate|1626947|
|Unhealthy for Sen...|  67552|
+--------------------+-------+



In [22]:
pm25_impact = (
    trips
    .filter(
        F.col("pm25_hourly_avg").isNotNull() &
        (F.col("trip_distance") > 0) &
        F.col("total_amount").isNotNull()
    )
    .withColumn(
        "aqi_category",
        F.when(F.col("pm25_hourly_avg") <= 12.0, "Good")
         .when(F.col("pm25_hourly_avg") <= 35.4, "Moderate")
         .when(
             F.col("pm25_hourly_avg") <= 55.4,
             "Unhealthy for Sensitive Groups"
         )
         .otherwise("Very Unhealthy")
    )
    .groupBy("aqi_category")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("pm25_hourly_avg"), 2).alias("avg_pm25"),
        F.round(F.avg("trip_distance"), 3).alias("avg_distance"),
        F.round(F.avg("total_amount"), 2).alias("avg_fare")
    )
    .orderBy("aqi_category")
)

pm25_impact.show(truncate=False)


+------------------------------+-----------+--------+------------+--------+
|aqi_category                  |total_trips|avg_pm25|avg_distance|avg_fare|
+------------------------------+-----------+--------+------------+--------+
|Good                          |6746309    |5.54    |3.489       |27.77   |
|Moderate                      |1626947    |19.46   |3.228       |27.6    |
|Unhealthy for Sensitive Groups|67552      |40.07   |3.807       |26.07   |
+------------------------------+-----------+--------+------------+--------+



In [23]:
pm25_monthly = (
    trips
    .filter(F.col("pm25_hourly_avg").isNotNull())
    .groupBy("year", "month")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("pm25_hourly_avg"), 2).alias("avg_pm25"),
        F.round(F.min("pm25_hourly_avg"), 2).alias("min_pm25"),
        F.round(F.max("pm25_hourly_avg"), 2).alias("max_pm25")
    )
    .orderBy("year", "month")
)

pm25_monthly.show()


+----+-----+-----------+--------+--------+--------+
|year|month|total_trips|avg_pm25|min_pm25|max_pm25|
+----+-----+-----------+--------+--------+--------+
|2024|    1|    2696637|    7.83|     1.0|    31.4|
|2024|    2|    2713394|    9.84|     1.5|    45.6|
|2024|    3|    3030775|    7.88|     0.7|    42.8|
|2024|    4|          2|     6.0|     6.0|     6.0|
+----+-----+-----------+--------+--------+--------+



In [24]:
print("=== PM2.5 MONTHLY PRODUCT ===")
print("Rows:", pm25_monthly.count())

print("\n=== NULL CHECK ===")
pm25_monthly.select(
    F.sum(F.col("year").isNull().cast("int")).alias("null_year"),
    F.sum(F.col("month").isNull().cast("int")).alias("null_month"),
    F.sum(F.col("total_trips").isNull().cast("int")).alias("null_trips"),
    F.sum(F.col("avg_pm25").isNull().cast("int")).alias("null_avg_pm25")
).show()

print("\n=== TOTAL TRIPS ===")
pm25_monthly.agg(
    F.sum("total_trips").alias("product_total")
).show()

print("\n=== SAMPLE ===")
pm25_monthly.show()


=== PM2.5 MONTHLY PRODUCT ===
Rows: 4

=== NULL CHECK ===
+---------+----------+----------+-------------+
|null_year|null_month|null_trips|null_avg_pm25|
+---------+----------+----------+-------------+
|        0|         0|         0|            0|
+---------+----------+----------+-------------+


=== TOTAL TRIPS ===
+-------------+
|product_total|
+-------------+
|      8440808|
+-------------+


=== SAMPLE ===
+----+-----+-----------+--------+--------+--------+
|year|month|total_trips|avg_pm25|min_pm25|max_pm25|
+----+-----+-----------+--------+--------+--------+
|2024|    1|    2696637|    7.83|     1.0|    31.4|
|2024|    2|    2713394|    9.84|     1.5|    45.6|
|2024|    3|    3030775|    7.88|     0.7|    42.8|
|2024|    4|          2|     6.0|     6.0|     6.0|
+----+-----+-----------+--------+--------+--------+



In [25]:
print("daily_mobility:", daily_mobility)
print("BASE:", BASE)


daily_mobility: DataFrame[trip_date: date, year: int, month: int, pickup_location_id: int, pickup_zone: string, pickup_borough: string, total_trips: bigint, total_distance: double, avg_distance: double, total_revenue: double, avg_fare: double]
BASE: f:\Sem 3A\Data-intensive Computing\urban-data-platform\data


In [30]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import os

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

daily_path = os.path.join(BASE, "data", "daily_mobility")

schema_version = "1.0"
data_source = "integrated_trips"

creation_time = datetime.now()
refresh_time = creation_time

# --------------------------------------------------
# 2. Build Daily Mobility Summary
# --------------------------------------------------

daily_mobility = (
    trips
    .filter(
        F.col("pickup_datetime").isNotNull() &
        F.col("trip_distance").isNotNull() &
        F.col("total_amount").isNotNull()
    )
    .withColumn(
        "trip_date",
        F.to_date("pickup_datetime")
    )
    .groupBy(
        "trip_date",
        "year",
        "month",
        "pickup_location_id",
        "pickup_zone",
        "pickup_borough"
    )
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("trip_distance").alias("total_distance"),
        F.avg("trip_distance").alias("avg_distance"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_fare")
    )
)

# --------------------------------------------------
# 3. Add metadata
# --------------------------------------------------

daily_mobility = (
    daily_mobility
    .withColumn("data_source", F.lit(data_source))
    .withColumn("creation_time", F.lit(creation_time))
    .withColumn("refresh_time", F.lit(refresh_time))
    .withColumn("schema_version", F.lit(schema_version))
)

# --------------------------------------------------
# 4. Write as Delta
# --------------------------------------------------

(
    daily_mobility
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year", "month")
    .save(daily_path)
)

print(f"Daily Mobility Summary saved to: {daily_path}")


Daily Mobility Summary saved to: f:\Sem 3A\Data-intensive Computing\urban-data-platform\data\data\daily_mobility


In [31]:
daily_test = spark.read.format("delta").load(daily_path)

daily_test.printSchema()

print("Rows:", daily_test.count())

daily_test.show(10, truncate=False)


root
 |-- trip_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- total_trips: long (nullable = true)
 |-- total_distance: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- data_source: string (nullable = true)
 |-- creation_time: timestamp (nullable = true)
 |-- refresh_time: timestamp (nullable = true)
 |-- schema_version: string (nullable = true)

Rows: 19793
+----------+----+-----+------------------+-----------------------------+--------------+-----------+------------------+------------------+------------------+------------------+----------------+--------------------------+--------------------------+--------------+
|trip_date |year|month|pickup_location_id|pickup_zone             

In [12]:
# ============================================================
# PRODUCT 1 – SOURCE RECONCILIATION
# ============================================================

print("=== SOURCE VS PRODUCT ===")

source_count = trips.count()
product_count = daily_mobility.agg(
    F.sum("total_trips").alias("trip_count")
).first()["trip_count"]

print(f"Source trip count : {source_count:,}")
print(f"Product trip count: {product_count:,}")
print(f"Match             : {source_count == product_count}")

print("\n=== SOURCE VS PRODUCT REVENUE ===")

source_revenue = trips.agg(
    F.sum("total_amount").alias("total_revenue")
).first()["total_revenue"]

product_revenue = daily_mobility.agg(
    F.sum("total_revenue").alias("total_revenue")
).first()["total_revenue"]

print(f"Source revenue : {source_revenue:,.2f}")
print(f"Product revenue: {product_revenue:,.2f}")
print(
    f"Difference     : "
    f"{abs(source_revenue - product_revenue):,.6f}"
)

print("\n=== SOURCE VS PRODUCT DISTANCE ===")

source_distance = trips.agg(
    F.sum("trip_distance").alias("total_distance")
).first()["total_distance"]

product_distance = daily_mobility.agg(
    F.sum("total_distance").alias("total_distance")
).first()["total_distance"]

print(f"Source distance : {source_distance:,.3f}")
print(f"Product distance: {product_distance:,.3f}")
print(
    f"Difference      : "
    f"{abs(source_distance - product_distance):,.6f}"
)


=== SOURCE VS PRODUCT ===
Source trip count : 8,480,836
Product trip count: 8,480,836
Match             : True

=== SOURCE VS PRODUCT REVENUE ===
Source revenue : 235,199,663.48
Product revenue: 235,199,663.48
Difference     : 0.000242

=== SOURCE VS PRODUCT DISTANCE ===
Source distance : 29,186,498.160
Product distance: 29,186,498.160
Difference      : 0.000001


Daily Mobility Summary preserves the complete trip population and correctly aggregates distance and revenue.

In [29]:

import os, sys
from datetime import datetime, timezone

os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
sys.path.insert(0, '..')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

In [27]:

# ---------------------------------------------------------------------------
# 2.  Load integrated dataset
# ---------------------------------------------------------------------------
# -- change this to your absolute path --
BASE = r'f:\Sem 3A\Data-intensive Computing\urban-data-platform\data'

trips = spark.read.format('delta').load(f'{BASE}/gold/integrated_taxi_trips')
trips.createOrReplaceTempView('trips')
print('Trip count:', trips.count())

Trip count: 8480836


In [30]:

# ---------------------------------------------------------------------------
# 3.  Shared metadata helper
# ---------------------------------------------------------------------------
SOURCE_TABLE  = 'gold/integrated_taxi_trips'
NOW           = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')

def add_metadata(df, schema_version: str):
    """Attach standard metadata columns to any product DataFrame."""
    return (
        df
        .withColumn('_data_source',    F.lit(SOURCE_TABLE))
        .withColumn('_schema_version', F.lit(schema_version))
        .withColumn('_created_at',     F.lit(NOW))   # treat first run as creation
        .withColumn('_refreshed_at',   F.lit(NOW))
    )

def save_product(df, name: str, partition_cols=None):
    """
    Write a data product as a Delta table under BASE/gold/products/<name>.
    Overwrites the existing data on each refresh (full refresh pattern).
    """
    out_path = f'{BASE}/gold/products/{name}'
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(out_path)
    print(f'  ✓ Saved {name}  ({df.count()} rows)  →  {out_path}')
    return out_path

In [36]:
trips = spark.read.format('delta').load(f'{BASE}/gold/integrated_taxi_trips')
trips.schema

StructType([StructField('vendor_id', IntegerType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('passenger_count', IntegerType(), True), StructField('trip_distance', DoubleType(), True), StructField('rate_code_id', IntegerType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('pickup_location_id', IntegerType(), True), StructField('dropoff_location_id', IntegerType(), True), StructField('payment_type', IntegerType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('year

In [37]:
# ===========================================================================
# PRODUCT 1 – Daily Mobility Summary
# ===========================================================================
# WHO:     City planners, transport operations teams
# WHY:     Tracks daily trip volume, average distance, and fare revenue per
#          zone.  Useful for detecting anomalies, planning driver allocation,
#          and monitoring seasonal patterns.
# WHY MAT: Aggregating 10M+ trips per day is expensive.  Pre-computing the
#          summary enables sub-second dashboard queries.
# ---------------------------------------------------------------------------
print('\n=== Product 1: daily_mobility_summary ===')
 
daily_mobility = spark.sql("""
    SELECT
        CAST(pickup_datetime AS DATE)       AS trip_date,
        year,
        month,
        pickup_location_id                  AS zone_id,
        pickup_zone,
        pickup_borough,
        COUNT(*)                            AS total_trips,
        ROUND(SUM(trip_distance),   2)      AS total_distance_km,
        ROUND(AVG(trip_distance),   3)      AS avg_distance_km,
        ROUND(SUM(total_amount),    2)      AS total_revenue,
        ROUND(AVG(total_amount),    2)      AS avg_fare
    FROM trips
    WHERE pickup_datetime IS NOT NULL
    GROUP BY
        CAST(pickup_datetime AS DATE),
        year, month,
        pickup_location_id, pickup_zone, pickup_borough
    ORDER BY trip_date, total_trips DESC
""")
 
daily_mobility = add_metadata(daily_mobility, schema_version='1.0.0')
save_product(daily_mobility, 'daily_mobility_summary', partition_cols=['year', 'month'])
 
daily_mobility.show(5, truncate=False)



=== Product 1: daily_mobility_summary ===
  ✓ Saved daily_mobility_summary  (19793 rows)  →  f:\Sem 3A\Data-intensive Computing\urban-data-platform\data/gold/products/daily_mobility_summary
+----------+----+-----+-------+---------------+--------------+-----------+-----------------+---------------+-------------+--------+--------------------------+---------------+-----------------------+-----------------------+
|trip_date |year|month|zone_id|pickup_zone    |pickup_borough|total_trips|total_distance_km|avg_distance_km|total_revenue|avg_fare|_data_source              |_schema_version|_created_at            |_refreshed_at          |
+----------+----+-----+-------+---------------+--------------+-----------+-----------------+---------------+-------------+--------+--------------------------+---------------+-----------------------+-----------------------+
|2002-12-31|2002|12   |50     |Clinton West   |Manhattan     |1          |1.4              |1.4            |18.0         |18.0    |gold/inte

In [38]:
 
# ===========================================================================
# PRODUCT 2 – Weather Impact Summary
# ===========================================================================
# WHO:     Data scientists, transport analysts studying demand elasticity
# WHY:     Quantifies how temperature and weather conditions affect trip
#          distance, trip count, and fare.  Enables weather-adjusted
#          demand forecasting.
# WHY MAT: Joining and bucketing all trips on weather every query is costly.
#          This product collapses that to a small lookup-sized table.
# ---------------------------------------------------------------------------
print('\n=== Product 2: weather_impact_summary ===')
 
weather_impact = spark.sql("""
    SELECT
        CASE
            WHEN temp < 5  THEN 'Cold  (<5°C)'
            WHEN temp < 20 THEN 'Mild  (5–20°C)'
            ELSE                'Hot   (>20°C)'
        END                                 AS temp_bucket,
        condition_code,
        COUNT(*)                            AS total_trips,
        ROUND(AVG(trip_distance),   3)      AS avg_distance_km,
        ROUND(AVG(total_amount),    2)      AS avg_fare,
        ROUND(STDDEV(trip_distance), 3)     AS stddev_distance
    FROM trips
    WHERE trip_distance > 0
      AND temp IS NOT NULL
    GROUP BY temp_bucket, condition_code
    ORDER BY temp_bucket, total_trips DESC
""")
 
weather_impact = add_metadata(weather_impact, schema_version='1.0.0')
save_product(weather_impact, 'weather_impact_summary')
 
weather_impact.show(truncate=False)



=== Product 2: weather_impact_summary ===
  ✓ Saved weather_impact_summary  (25 rows)  →  f:\Sem 3A\Data-intensive Computing\urban-data-platform\data/gold/products/weather_impact_summary
+--------------+--------------+-----------+---------------+--------+---------------+--------------------------+---------------+-----------------------+-----------------------+
|temp_bucket   |condition_code|total_trips|avg_distance_km|avg_fare|stddev_distance|_data_source              |_schema_version|_created_at            |_refreshed_at          |
+--------------+--------------+-----------+---------------+--------+---------------+--------------------------+---------------+-----------------------+-----------------------+
|Cold  (<5°C)  |2             |1636589    |3.572          |27.7    |104.025        |gold/integrated_taxi_trips|1.0.0          |2026-09-17 11:07:15 UTC|2026-09-17 11:07:15 UTC|
|Cold  (<5°C)  |4             |802877     |3.239          |26.95   |36.171         |gold/integrated_taxi_tri

In [39]:

# ===========================================================================
# PRODUCT 3 – Air Quality Impact Summary
# ===========================================================================
# WHO:     Public-health researchers, environmental agencies, city analysts
# WHY:     Shows whether poor air quality correlates with reduced taxi demand
#          (people staying indoors) or increased demand (avoiding walking).
#          Breaks demand down by AQI category and month.
# WHY MAT: The PM2.5 filter, CASE bucketing, and monthly aggregation across
#          millions of rows is expensive to repeat ad-hoc.
# ---------------------------------------------------------------------------
print('\n=== Product 3: air_quality_impact_summary ===')
 
air_quality_impact = spark.sql("""
    SELECT
        year,
        month,
        CASE
            WHEN pm25_hourly_avg <= 12  THEN 'Good'
            WHEN pm25_hourly_avg <= 35  THEN 'Moderate'
            WHEN pm25_hourly_avg <= 55  THEN 'Unhealthy for Sensitive Groups'
            WHEN pm25_hourly_avg <= 150 THEN 'Unhealthy'
            ELSE                             'Very Unhealthy'
        END                                 AS aqi_category,
        COUNT(*)                            AS total_trips,
        ROUND(AVG(pm25_hourly_avg),  1)     AS avg_pm25,
        ROUND(AVG(trip_distance),    3)     AS avg_distance_km,
        ROUND(AVG(total_amount),     2)     AS avg_fare
    FROM trips
    WHERE pm25_hourly_avg IS NOT NULL
    GROUP BY year, month, aqi_category
    ORDER BY year, month, avg_pm25
""")
 
air_quality_impact = add_metadata(air_quality_impact, schema_version='1.0.0')
save_product(air_quality_impact, 'air_quality_impact_summary', partition_cols=['year'])
 
air_quality_impact.show(truncate=False)


=== Product 3: air_quality_impact_summary ===
  ✓ Saved air_quality_impact_summary  (9 rows)  →  f:\Sem 3A\Data-intensive Computing\urban-data-platform\data/gold/products/air_quality_impact_summary
+----+-----+------------------------------+-----------+--------+---------------+--------+--------------------------+---------------+-----------------------+-----------------------+
|year|month|aqi_category                  |total_trips|avg_pm25|avg_distance_km|avg_fare|_data_source              |_schema_version|_created_at            |_refreshed_at          |
+----+-----+------------------------------+-----------+--------+---------------+--------+--------------------------+---------------+-----------------------+-----------------------+
|2024|1    |Good                          |2187741    |5.6     |3.335          |27.46   |gold/integrated_taxi_trips|1.0.0          |2026-09-17 11:07:15 UTC|2026-09-17 11:07:15 UTC|
|2024|1    |Moderate                      |508896     |17.5    |3.164        

In [40]:

# ===========================================================================
# PRODUCT 4 – Taxi Zone Statistics
# ===========================================================================
# WHO:     Fleet managers, zone-level operations teams, urban planners
# WHY:     Summarises per-zone demand stability across weather conditions.
#          High-variance zones may need dynamic pricing or extra coverage
#          during bad weather.
# WHY MAT: Requires a two-level aggregation (trips → zone+weather, then
#          → zone statistics) which is expensive on raw data every time.
# ---------------------------------------------------------------------------
print('\n=== Product 4: taxi_zone_statistics ===')
 
taxi_zone_stats = spark.sql("""
    WITH zone_weather AS (
        SELECT
            pickup_location_id              AS zone_id,
            pickup_zone,
            pickup_borough,
            condition_code,
            COUNT(*)                        AS trip_count,
            ROUND(AVG(trip_distance), 3)    AS avg_distance_km
        FROM trips
        WHERE condition_code IS NOT NULL
        GROUP BY pickup_location_id, pickup_zone, pickup_borough, condition_code
    )
    SELECT
        zone_id,
        pickup_zone,
        pickup_borough,
        ROUND(AVG(trip_count),             1) AS avg_demand_per_condition,
        ROUND(STDDEV(trip_count),          1) AS demand_stddev,
        ROUND(MAX(trip_count),             0) AS peak_demand,
        ROUND(MIN(trip_count),             0) AS min_demand,
        ROUND(MAX(trip_count) - MIN(trip_count), 0) AS demand_range,
        COUNT(DISTINCT condition_code)         AS num_weather_conditions,
        ROUND(AVG(avg_distance_km),        3)  AS overall_avg_distance_km
    FROM zone_weather
    GROUP BY zone_id, pickup_zone, pickup_borough
    HAVING COUNT(DISTINCT condition_code) >= 3
    ORDER BY demand_stddev DESC
""")
 
taxi_zone_stats = add_metadata(taxi_zone_stats, schema_version='1.0.0')
save_product(taxi_zone_stats, 'taxi_zone_statistics')
 
taxi_zone_stats.show(20, truncate=False)
 



=== Product 4: taxi_zone_statistics ===
  ✓ Saved taxi_zone_statistics  (250 rows)  →  f:\Sem 3A\Data-intensive Computing\urban-data-platform\data/gold/products/taxi_zone_statistics
+-------+----------------------------+--------------+------------------------+-------------+-----------+----------+------------+----------------------+-----------------------+--------------------------+---------------+-----------------------+-----------------------+
|zone_id|pickup_zone                 |pickup_borough|avg_demand_per_condition|demand_stddev|peak_demand|min_demand|demand_range|num_weather_conditions|overall_avg_distance_km|_data_source              |_schema_version|_created_at            |_refreshed_at          |
+-------+----------------------------+--------------+------------------------+-------------+-----------+----------+------------+----------------------+-----------------------+--------------------------+---------------+-----------------------+-----------------------+
|161    |Midtown

In [41]:


# ===========================================================================
# 4.  Verify: reload each product from Delta and spot-check row counts
# ===========================================================================
print('\n=== Verification: reloading products from Delta ===')
 
products = [
    'daily_mobility_summary',
    'weather_impact_summary',
    'air_quality_impact_summary',
    'taxi_zone_statistics',
]
 
for name in products:
    path = f'{BASE}/gold/products/{name}'
    df   = spark.read.format('delta').load(path)
    print(f'  {name:<35}  rows={df.count():>8}  '
          f'cols={len(df.columns)}  '
          f'schema_version={df.select("_schema_version").first()[0]}')
 
print('\nAll data products written successfully.')
spark.stop()



=== Verification: reloading products from Delta ===
  daily_mobility_summary               rows=   19793  cols=15  schema_version=1.0.0
  weather_impact_summary               rows=      25  cols=10  schema_version=1.0.0
  air_quality_impact_summary           rows=       9  cols=11  schema_version=1.0.0
  taxi_zone_statistics                 rows=     250  cols=14  schema_version=1.0.0

All data products written successfully.


In [ ]:
# %% [markdown]
# # Task 5 – Platform Evaluation
# Measures execution time before/after each optimization, verifies result
# correctness, inspects EXPLAIN plans, and reports storage overhead of the
# four analytical data products.

# %% ── Session setup ────────────────────────────────────────────────────────
import os, sys, time, statistics, json
from datetime import datetime, timezone

os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
sys.path.insert(0, '..')

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName('week2-task5-evaluation')
    .master('local[*]')
    .config('spark.sql.extensions',           'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions',   '8')
    .config('spark.sql.session.timeZone',     'America/New_York')
    .config('spark.driver.memory',            '4g')
    .config('spark.executor.memory',          '4g')
    .config('spark.sql.adaptive.enabled',     'true')   # default on; toggled per experiment
    .config('spark.sql.autoBroadcastJoinThreshold', '104857600')  # 100 MB
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print('Spark ready:', spark.version)

# %% ── Paths ─────────────────────────────────────────────────────────────────
BASE = r'f:\Sem 3A\Data-intensive Computing\urban-data-platform\data'

trips = spark.read.format('delta').load(f'{BASE}/gold/integrated_taxi_trips')
trips.createOrReplaceTempView('trips')
print('Total trips:', trips.count())

# %% ── Benchmark helpers ──────────────────────────────────────────────────────
def bench(sql_or_fn, n=3, label=''):
    """
    Run a SQL string (or zero-arg callable) n times.
    Returns median wall-clock seconds and prints all run times.
    """
    times = []
    for _ in range(n):
        t0 = time.time()
        if callable(sql_or_fn):
            sql_or_fn()
        else:
            spark.sql(sql_or_fn).collect()
        times.append(round(time.time() - t0, 3))
    med = round(statistics.median(times), 3)
    print(f'  {label:<40}  runs={times}  median={med}s')
    return med


def verify_same(sql1, sql2, label=''):
    """Assert two SQL queries return identical result sets."""
    df1 = spark.sql(sql1)
    df2 = spark.sql(sql2)
    diff = df1.subtract(df2).count() + df2.subtract(df1).count()
    status = '✓ identical' if diff == 0 else f'✗ DIFFER by {diff} rows'
    print(f'  Result check [{label}]: {status}')
    return diff == 0


def print_explain(sql, label=''):
    print(f'\n── EXPLAIN FORMATTED: {label} ──')
    spark.sql(sql).explain('formatted')


results = {}   # { label: median_seconds }

# ============================================================================
# EXP 1 – CACHING  (Q1: monthly demand by zone)
# ============================================================================
# %% [markdown]
# ## Experiment 1 – Caching (Q1)

# %%
Q1 = """
    SELECT year, month, pickup_location_id AS zone_id, pickup_zone,
           COUNT(*) AS trip_count
    FROM {view}
    WHERE year >= 2024
    GROUP BY year, month, pickup_location_id, pickup_zone
    ORDER BY year, month, trip_count DESC
"""

print('=== EXP 1: Caching ===')

# -- baseline (no cache)
results['Q1 no cache'] = bench(Q1.format(view='trips'), label='Q1 no cache')

# -- populate cache
cached = spark.sql('SELECT * FROM trips WHERE year >= 2024').cache()
cached.createOrReplaceTempView('trips_cached')
cached.count()   # materialise
print('  Cache materialised.')

results['Q1 cached'] = bench(Q1.format(view='trips_cached'), label='Q1 cached')

verify_same(Q1.format(view='trips'), Q1.format(view='trips_cached'), label='Q1 cache')

print_explain(Q1.format(view='trips'),        label='Q1 no cache')
print_explain(Q1.format(view='trips_cached'), label='Q1 cached')

# ============================================================================
# EXP 2 – PARTITION PRUNING  (Q6: monthly trend)
# ============================================================================
# %% [markdown]
# ## Experiment 2 – Partition Pruning (Q6)

# %%
Q6_no_prune = """
    SELECT year, month, COUNT(*) AS trip_count
    FROM trips
    GROUP BY year, month
    ORDER BY year, month
"""

Q6_pruned = """
    SELECT year, month, COUNT(*) AS trip_count
    FROM trips
    WHERE year = 2024
    GROUP BY year, month
    ORDER BY year, month
"""

print('\n=== EXP 2: Partition Pruning ===')
results['Q6 no prune'] = bench(Q6_no_prune, label='Q6 no pruning')
results['Q6 pruned']   = bench(Q6_pruned,   label='Q6 pruned (year=2024)')

# Correctness note: pruned returns a subset, so we verify the pruned rows
# all appear in the full result (not strict equality).
full   = spark.sql(Q6_no_prune)
pruned = spark.sql(Q6_pruned)
missing = pruned.subtract(full).count()
print(f'  Result check [Q6 prune]: pruned rows present in full result → '
      f'{"✓" if missing == 0 else "✗ " + str(missing) + " missing"}')

print_explain(Q6_no_prune, label='Q6 no pruning')
print_explain(Q6_pruned,   label='Q6 pruned')

# ============================================================================
# EXP 3 – BROADCAST JOIN  (Q2: weather conditions)
# ============================================================================
# %% [markdown]
# ## Experiment 3 – Broadcast Join (Q2)
#
# The integrated table already contains weather columns, so there is no
# separate join needed for Q2 as written.  We demonstrate the broadcast
# technique on a realistic scenario: re-joining trips with a small
# condition_labels lookup (built inline) to attach a human-readable label.

# %%
# Build a tiny condition lookup from the data itself
spark.sql("""
    SELECT DISTINCT condition_code,
           CONCAT('Condition_', CAST(condition_code AS STRING)) AS condition_name
    FROM trips
    WHERE condition_code IS NOT NULL
""").createOrReplaceTempView('condition_labels')

Q2_no_hint = """
    SELECT t.condition_code, cl.condition_name,
           ROUND(AVG(t.trip_distance), 3) AS avg_dist,
           COUNT(*) AS trip_count
    FROM trips t
    LEFT JOIN condition_labels cl ON t.condition_code = cl.condition_code
    WHERE t.trip_distance > 0
    GROUP BY t.condition_code, cl.condition_name
    ORDER BY trip_count DESC
"""

Q2_broadcast = """
    SELECT /*+ BROADCAST(cl) */ t.condition_code, cl.condition_name,
           ROUND(AVG(t.trip_distance), 3) AS avg_dist,
           COUNT(*) AS trip_count
    FROM trips t
    LEFT JOIN condition_labels cl ON t.condition_code = cl.condition_code
    WHERE t.trip_distance > 0
    GROUP BY t.condition_code, cl.condition_name
    ORDER BY trip_count DESC
"""

print('\n=== EXP 3: Broadcast Join ===')

# Force sort-merge baseline
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
results['Q2 sort-merge'] = bench(Q2_no_hint, label='Q2 forced sort-merge join')

# Re-enable auto broadcast
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '104857600')
results['Q2 auto broadcast']    = bench(Q2_no_hint,   label='Q2 auto broadcast')
results['Q2 explicit broadcast'] = bench(Q2_broadcast, label='Q2 explicit BROADCAST hint')

verify_same(Q2_no_hint, Q2_broadcast, label='Q2 broadcast')

print_explain(Q2_no_hint,   label='Q2 no hint (auto)')
print_explain(Q2_broadcast, label='Q2 explicit broadcast')

# ============================================================================
# EXP 4 – ADAPTIVE QUERY EXECUTION  (Q3: air quality vs demand)
# ============================================================================
# %% [markdown]
# ## Experiment 4 – Adaptive Query Execution (Q3)

# %%
Q3 = """
    SELECT year, month,
        CASE
            WHEN pm25_hourly_avg <= 12  THEN 'Good'
            WHEN pm25_hourly_avg <= 35  THEN 'Moderate'
            WHEN pm25_hourly_avg <= 55  THEN 'Unhealthy for Sensitive Groups'
            WHEN pm25_hourly_avg <= 150 THEN 'Unhealthy'
            ELSE 'Very Unhealthy'
        END AS aqi_category,
        COUNT(*) AS trip_count,
        ROUND(AVG(pm25_hourly_avg), 1) AS avg_pm25
    FROM trips
    WHERE pm25_hourly_avg IS NOT NULL
    GROUP BY year, month, aqi_category
    ORDER BY year, month
"""

print('\n=== EXP 4: Adaptive Query Execution ===')

spark.conf.set('spark.sql.adaptive.enabled', 'false')
results['Q3 AQE off'] = bench(Q3, label='Q3 AQE disabled')

spark.conf.set('spark.sql.adaptive.enabled', 'true')
results['Q3 AQE on']  = bench(Q3, label='Q3 AQE enabled')

verify_same(Q3, Q3, label='Q3 AQE (same query)')   # always identical; proves no side-effect

spark.conf.set('spark.sql.adaptive.enabled', 'false')
print_explain(Q3, label='Q3 AQE off')
spark.conf.set('spark.sql.adaptive.enabled', 'true')
print_explain(Q3, label='Q3 AQE on')

# ============================================================================
# EXP 5 – DATA PRODUCT QUERY vs RAW  (Q4: zone variance)
# ============================================================================
# %% [markdown]
# ## Experiment 5 – Querying pre-materialised data product vs raw (Q4)

# %%
Q4_raw = """
    WITH zone_weather AS (
        SELECT pickup_location_id AS zone_id, pickup_zone, condition_code,
               COUNT(*) AS trip_count
        FROM trips
        WHERE condition_code IS NOT NULL
        GROUP BY pickup_location_id, pickup_zone, condition_code
    )
    SELECT zone_id, pickup_zone,
           ROUND(STDDEV(trip_count), 1) AS demand_stddev,
           ROUND(AVG(trip_count),    1) AS avg_demand
    FROM zone_weather
    GROUP BY zone_id, pickup_zone
    HAVING COUNT(DISTINCT condition_code) >= 3
    ORDER BY demand_stddev DESC
    LIMIT 20
"""

# Load the pre-built product
zone_product = spark.read.format('delta').load(f'{BASE}/gold/products/taxi_zone_statistics')
zone_product.createOrReplaceTempView('taxi_zone_statistics')

Q4_product = """
    SELECT zone_id, pickup_zone, demand_stddev,
           avg_demand_per_condition AS avg_demand
    FROM taxi_zone_statistics
    ORDER BY demand_stddev DESC
    LIMIT 20
"""

print('\n=== EXP 5: Data Product vs Raw (Q4) ===')
results['Q4 raw']     = bench(Q4_raw,     label='Q4 computed on raw trips')
results['Q4 product'] = bench(Q4_product, label='Q4 from materialised product')

# Verify top-20 zone_ids match (order-independent)
raw_ids  = set(r.zone_id for r in spark.sql(Q4_raw).collect())
prod_ids = set(r.zone_id for r in spark.sql(Q4_product).collect())
overlap  = len(raw_ids & prod_ids)
print(f'  Top-20 zone_id overlap: {overlap}/20 '
      f'{"✓" if overlap >= 18 else "⚠ check data"}')

# ============================================================================
# STORAGE OVERHEAD of data products
# ============================================================================
# %% [markdown]
# ## Storage Overhead of Analytical Data Products

# %%
import os

def dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, f))
            except OSError:
                pass
    return round(total / 1_048_576, 2)

products = [
    'daily_mobility_summary',
    'weather_impact_summary',
    'air_quality_impact_summary',
    'taxi_zone_statistics',
]

print('\n=== Storage Overhead ===')
print(f'  {"Product":<35}  {"Size (MB)":>10}')
print('  ' + '-' * 48)
total_mb = 0
for name in products:
    path = f'{BASE}/gold/products/{name}'
    mb   = dir_size_mb(path)
    total_mb += mb
    print(f'  {name:<35}  {mb:>10.2f}')
print(f'  {"TOTAL":<35}  {round(total_mb,2):>10.2f}')

raw_mb = dir_size_mb(f'{BASE}/gold/integrated_taxi_trips')
print(f'\n  Raw integrated table size : {raw_mb:.2f} MB')
print(f'  Products overhead ratio   : {round(total_mb/raw_mb*100,1)}% of raw')

# SUMMARY TABLE
print('\n' + '='*60)
print(f'  {"Experiment":<40} {"Time (s)":>8}')
print('  ' + '-'*50)
for label, t in results.items():
    print(f'  {label:<40} {t:>8.3f}')

print('\n── Speedups ──')
pairs = [
    ('Q1 no cache',    'Q1 cached',            'Caching (Q1)'),
    ('Q6 no prune',    'Q6 pruned',             'Partition pruning (Q6)'),
    ('Q2 sort-merge',  'Q2 explicit broadcast', 'Broadcast join (Q2)'),
    ('Q3 AQE off',     'Q3 AQE on',             'AQE (Q3)'),
    ('Q4 raw',         'Q4 product',            'Materialised product (Q4)'),
]
for base_key, opt_key, name in pairs:
    if base_key in results and opt_key in results:
        speedup = round(results[base_key] / results[opt_key], 2)
        saved   = round(results[base_key] - results[opt_key], 3)
        print(f'  {name:<35}  {results[base_key]}s → {results[opt_key]}s  '
              f'({speedup}x speedup, {saved}s saved)')

print('\nTask 5 evaluation complete.')
spark.stop()

Spark ready: 3.5.9
Total trips: 8480836
=== EXP 1: Caching ===
  Q1 no cache                               runs=[1.854, 0.786, 0.721]  median=0.786s
  Cache materialised.
  Q1 cached                                 runs=[0.711, 0.647, 0.561]  median=0.647s
  Result check [Q1 cache]: ✓ identical

── EXPLAIN FORMATTED: Q1 no cache ──
== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Exchange (3)
            +- HashAggregate (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [pickup_location_id#20702, pickup_zone#20725, year#20714, month#20715]
Batched: true
Location: PreparedDeltaFileIndex [file:/f:/Sem 3A/Data-intensive Computing/urban-data-platform/data/gold/integrated_taxi_trips]
PartitionFilters: [isnotnull(year#20714), (year#20714 >= 2024)]
ReadSchema: struct<pickup_location_id:int,pickup_zone:string>

(2) HashAggregate
Input [4]: [pickup_location_id#20702, pickup_zone#20725, year#20714, month#20715]
